# Semana 12: Autovalores y Autovectores - Las Direcciones Clave
## Notebook Conceptual (NB1) - Visualizando Autovectores con Datos Sintéticos

### Propósito de la sesión
Descubrir que los **autovectores** encuentran las direcciones de máxima varianza de los datos, siendo la base del **Análisis de Componentes Principales (PCA)**. Visualizaremos una nube de puntos 2D con correlación, calcularemos su matriz de covarianza, obtendremos autovalores y autovectores, y observaremos cómo el primer autovector apunta en la dirección de máxima varianza.

### Objetivos de aprendizaje
*   Generar una nube de puntos 2D con correlación (elipsoide alargado).
*   Calcular la **matriz de covarianza** de los datos.
*   Obtener **autovalores ($\lambda$)** y **autovectores ($v$)** usando `np.linalg.eig`.
*   Visualizar los autovectores sobre la nube de puntos y verificar que el primero apunta en la dirección de máxima varianza.
*   Conectar estos conceptos con **PCA**, **LSA**, **PageRank** y espacios latentes en **modelos generativos**.

## Configuración Inicial

Importamos las librerías necesarias: NumPy para operaciones, Matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 8)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Generación de datos: Nube de puntos 2D con correlación

Generamos una nube de puntos que forme un elipsoide alargado, es decir, con una dirección de mayor varianza. Esto se logra multiplicando puntos aleatorios normales por una matriz de transformación.

In [ ]:
# Parámetros
n_samples = 500
mean = [0, 0]
cov = [[3, 2.5],   # matriz de covarianza (definida positiva)
       [2.5, 3]]   # alta correlación, elipse alargada

# Generamos datos
data = np.random.multivariate_normal(mean, cov, n_samples)

print(f"Shape de los datos: {data.shape}")
print(f"Media real: {np.mean(data, axis=0)}")
print(f"Covarianza real:\n{np.cov(data.T)}")

# Visualizamos
plt.figure(figsize=(8, 8))
plt.scatter(data[:, 0], data[:, 1], alpha=0.6, edgecolors='k')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('Nube de puntos 2D con correlación (elipsoide alargado)')
plt.axis('equal')
plt.grid(True)
plt.show()

---
## 2. Cálculo de la matriz de covarianza

La matriz de covarianza muestral se calcula como:

$$\Sigma = \frac{1}{n-1} \sum_{i=1}^{n} (\mathbf{x}_i - \bar{\mathbf{x}})(\mathbf{x}_i - \bar{\mathbf{x}})^\top$$

O de forma matricial (con datos centrados): $\Sigma = \frac{1}{n-1} \mathbf{X}_c^\top \mathbf{X}_c$

In [ ]:
# Centramos los datos (restamos la media)
data_centered = data - np.mean(data, axis=0)

# Calculamos la matriz de covarianza (dos formas)
cov_matrix = np.cov(data.T)
cov_matrix_manual = (data_centered.T @ data_centered) / (n_samples - 1)

print("Matriz de covarianza (np.cov):")
print(cov_matrix)
print("\nMatriz de covarianza (manual):")
print(cov_matrix_manual)

# Verificamos que son iguales (dentro de tolerancia numérica)
print(f"\nDiferencia máxima: {np.max(np.abs(cov_matrix - cov_matrix_manual)):.2e}")

---
## 3. Cálculo de autovalores y autovectores

Los autovalores $\lambda$ y autovectores $v$ de la matriz de covarianza satisfacen:

$$\Sigma v = \lambda v$$

Los autovalores representan la varianza a lo largo de la dirección de sus correspondientes autovectores.

In [ ]:
# Calculamos autovalores y autovectores
eigvals, eigvecs = np.linalg.eig(cov_matrix)

# Los autovectores son las columnas de eigvecs
print("Autovalores:", eigvals)
print("Autovectores (como columnas):\n", eigvecs)

# Ordenamos de mayor a menor autovalor (importante para PCA)
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("\n🔷 Autovalores ordenados:", eigvals)
print("Autovectores ordenados (columnas):\n", eigvecs)

# Verificamos la descomposición
for i in range(2):
    v = eigvecs[:, i]
    lam = eigvals[i]
    print(f"\nPara λ={lam:.4f}, v={v}:")
    print(f"  Σ·v = {cov_matrix @ v}")
    print(f"  λ·v = {lam * v}")

---
## 4. Visualización de los autovectores sobre la nube de puntos

Graficamos los autovectores (escalados por la raíz cuadrada de sus autovalores para representar la magnitud de la varianza) sobre la nube de puntos.

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(data[:, 0], data[:, 1], alpha=0.5, edgecolors='k', label='Datos')

# Dibujamos los autovectores (escalados por 2*√λ para que se vean bien)
mean_data = np.mean(data, axis=0)
for i in range(2):
    v = eigvecs[:, i]
    lam = eigvals[i]
    # Escalamos el vector por 2*√λ para representar la desviación estándar en esa dirección
    scale = 2 * np.sqrt(lam)
    plt.arrow(mean_data[0], mean_data[1], v[0]*scale, v[1]*scale,
              head_width=0.2, head_length=0.2, fc='red', ec='red',
              label=f'Autovector {i+1} (λ={lam:.2f})')

plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('Autovectores de la matriz de covarianza sobre los datos')
plt.legend()
plt.axis('equal')
plt.grid(True)
plt.show()

print("📌 El primer autovector (rojo) apunta en la dirección de máxima varianza.")
print(f"📌 Varianza explicada por componente 1: {eigvals[0]/np.sum(eigvals):.2%}")
print(f"📌 Varianza explicada por componente 2: {eigvals[1]/np.sum(eigvals):.2%}")

---
## 5. Proyección sobre los autovectores (PCA)

El **Análisis de Componentes Principales (PCA)** consiste en proyectar los datos sobre los autovectores (componentes principales) para reducir dimensionalidad.

In [ ]:
# Proyectamos sobre el primer autovector (1D)
v1 = eigvecs[:, 0]
proj_1d = data_centered @ v1  # coordenadas en la dirección de máxima varianza

# Reconstruimos los puntos en 2D a partir de la proyección 1D
reconstructed = np.outer(proj_1d, v1) + mean_data

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(data[:, 0], data[:, 1], alpha=0.5, label='Originales')
plt.scatter(reconstructed[:, 0], reconstructed[:, 1], alpha=0.5, c='red', label='Reconstruidos (1 componente)')
for i in range(len(data)):
    plt.plot([data[i,0], reconstructed[i,0]], [data[i,1], reconstructed[i,1]], 'k--', alpha=0.1)
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('Proyección sobre el primer autovector (PCA 1D)')
plt.legend()
plt.axis('equal')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(proj_1d, np.zeros_like(proj_1d), alpha=0.5)
plt.xlabel('Componente principal 1')
plt.title('Datos proyectados a 1D')
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Varianza retenida con 1 componente: {eigvals[0]/np.sum(eigvals):.2%}")

---
## 6. Conexiones IA

### 6.1. ML/CV: PCA (Principal Component Analysis)
PCA usa los autovectores de la matriz de covarianza para reducir dimensionalidad, proyectando los datos sobre las direcciones de máxima varianza. La varianza explicada por cada componente es su autovalor.

### 6.2. NLP: Latent Semantic Analysis (LSA)
LSA aplica SVD (relacionado con autovalores) a la matriz término-documento para encontrar "temas" latentes. Los autovectores representan direcciones semánticas.

### 6.3. GenAI: Espacios latentes interpretables
En modelos generativos (como GANs o VAEs), los autovectores del espacio latente pueden corresponder a direcciones interpretables (ej. "sonrisa" en un espacio de caras).

### 6.4. RL: PageRank
El algoritmo PageRank de Google calcula el autovector principal de la matriz de transición de la web (autovalor 1) para determinar la importancia de las páginas.

In [ ]:
# Ejemplo conceptual: PageRank (método de la potencia simplificado)
# Matriz de transición de ejemplo (pequeña web)
G = np.array([[0.0, 0.5, 0.5],
              [0.5, 0.0, 0.5],
              [0.5, 0.5, 0.0]])

# Aseguramos que las columnas sumen 1 (matriz estocástica)
G = G / G.sum(axis=0, keepdims=True)

# Método de la potencia para encontrar el autovector principal (λ≈1)
v = np.random.rand(3)
v = v / np.linalg.norm(v)
for _ in range(50):
    v = G @ v
    v = v / np.linalg.norm(v)

print("Autovector principal (PageRank aproximado):", v)
print("Suma de componentes:", v.sum())

# Normalizamos a suma 1 para obtener distribución de importancia
ranking = v / v.sum()
print("Ranking de páginas (importancia):", ranking)

---
## Ejercicios para el Estudiante

1.  **Diferentes correlaciones:** Genera datos con diferentes matrices de covarianza (por ejemplo, correlación negativa, círculo, etc.) y observa cómo cambian los autovectores.

2.  **PCA manual en Iris:** Carga el dataset Iris de sklearn y aplica PCA desde cero usando autovalores/autovectores (como en el notebook). Visualiza las dos primeras componentes.

3.  **Varianza explicada:** Para los datos generados, calcula la proporción de varianza explicada por cada componente. ¿Cuántas componentes necesitas para retener el 95% de la varianza?

4.  **SVD vs eig:** Investiga por qué en la práctica se prefiere usar SVD (`np.linalg.svd`) en lugar de autodescomposición para PCA. Implementa PCA usando SVD y compara resultados.

5.  **Reflexión:** En modelos generativos, ¿cómo crees que se podrían identificar direcciones interpretables en el espacio latente? ¿Qué relación tiene con autovectores?

---
## Conclusiones de la Sesión

*   Los **autovalores** y **autovectores** de la matriz de covarianza revelan las direcciones de máxima varianza de los datos.
*   El autovector con mayor autovalor apunta en la dirección donde los datos están más dispersos (máxima varianza).
*   Hemos visualizado estos autovectores sobre la nube de puntos, confirmando su interpretación geométrica.
*   El **PCA** proyecta los datos sobre estos autovectores para reducir dimensionalidad, reteniendo la mayor parte de la información (varianza).
*   Estas ideas se extienden a múltiples aplicaciones:
    *   **ML/CV:** PCA para reducción de dimensionalidad y visualización.
    *   **NLP:** LSA para análisis semántico latente.
    *   **GenAI:** Espacios latentes interpretables.
    *   **RL:** PageRank como problema de autovector dominante.

¡Ahora entiendes cómo los autovectores capturan las direcciones clave en los datos!